# Can we predict a warranty claim within a year?

Target: `claim_within_year`. Plan: check data quality, explore, then compare a baseline, logistic regression and gradient boosting using CV on a training split and one final look at a held-out test set. Metrics: ROC AUC and PR AUC (classes are imbalanced), plus performance at a chosen threshold.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv')
df.shape

## 1. Data quality

In [ ]:
print(df.isna().sum().to_string()); print('duplicate rows:',df.duplicated().sum(),' duplicate serials:',df.device_serial.duplicated().sum())
df.describe().T

In [ ]:
print('daily_hours > 24:',(df.daily_hours>24).sum(), ' <0:',(df.daily_hours<0).sum())
print('age <0:',(df.device_age_years<0).sum(),' price<=0:',(df.price_eur<=0).sum())
df.daily_hours.describe(percentiles=[.5,.9,.99])

`daily_hours` has values above 24, impossible for a daily average (likely entry errors). If there are few, I keep the rows and clip the feature at 24. `device_serial` is an identifier, so it is excluded from the features (checked below that it carries no signal).

In [ ]:
y=df.claim_within_year
print('base rate:',y.mean().round(4))
fig,ax=plt.subplots(1,4,figsize=(18,3.8))
for a,c in zip(ax[:3],['device_age_years','price_eur','daily_hours']):
    q=pd.qcut(df[c],10,duplicates='drop'); r=y.groupby(q,observed=True).mean()
    a.plot(range(len(r)),r.values,marker='o'); a.set_title('claim rate by '+c+' decile'); a.set_xlabel('decile'); a.axhline(y.mean(),ls='--',c='gray')
df.groupby('channel').claim_within_year.mean().plot.bar(ax=ax[3],title='claim rate by channel'); ax[3].axhline(y.mean(),ls='--',c='gray')
plt.tight_layout(); plt.show()

In [ ]:
print('AUC of serial alone:', round(roc_auc_score(y, df.device_serial),3))
print(df.groupby('channel').claim_within_year.agg(['mean','count']))

## 2. Modelling
Stratified 80/20 split; model comparison by 5-fold CV on the training part only; the test set is used once at the end.

In [ ]:
df['daily_hours']=df.daily_hours.clip(upper=24)
df['log_price']=np.log1p(df.price_eur)
num=['device_age_years','log_price','daily_hours']; cat=['channel']
X=df[num+cat]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED)
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),cat)])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic regression':Pipeline([('p',pre),('m',LogisticRegression(max_iter=1000))]),
 'gradient boosting':Pipeline([('p',pre),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=.05,max_iter=200,l2_regularization=1.0,random_state=SEED))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    a=cross_val_score(m,Xtr,ytr,cv=cv,scoring='roc_auc'); p=cross_val_score(m,Xtr,ytr,cv=cv,scoring='average_precision')
    rows.append((n,a.mean(),a.std(),p.mean(),p.std()))
pd.DataFrame(rows,columns=['model','cv_auc','auc_sd','cv_pr_auc','pr_sd']).round(3)

## 3. Held-out test performance

In [ ]:
res={}
for n,m in models.items():
    m.fit(Xtr,ytr); s=m.predict_proba(Xte)[:,1]; res[n]=s
    print(n.ljust(22), 'AUC=%.3f  PR-AUC=%.3f' % (roc_auc_score(yte,s), average_precision_score(yte,s)))
print('PR-AUC of a random model = base rate =',round(yte.mean(),3))
fig,ax=plt.subplots(1,2,figsize=(11,4))
for n,s in res.items():
    f,t,_=roc_curve(yte,s); ax[0].plot(f,t,label=n)
    p,r,_=precision_recall_curve(yte,s); ax[1].plot(r,p,label=n)
ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR'); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision'); ax[0].legend(); plt.show()

### Operating point and calibration
With a ~12.6% base rate accuracy is misleading (always predicting 'no claim' gives ~87%). Below: flag the riskiest 20% of warranties, look at the lift, then calibration by decile.

In [ ]:
s=res['logistic regression']; thr=np.quantile(s,.8); pred=(s>=thr).astype(int)
print(confusion_matrix(yte,pred)); print(classification_report(yte,pred,digits=3))
print('claim rate in top 20%:',round(yte[pred==1].mean(),3),' vs rest:',round(yte[pred==0].mean(),3))
dec=pd.qcut(s,10,labels=False,duplicates='drop')
cal=pd.DataFrame({'pred':s,'actual':yte.values,'d':dec}).groupby('d').mean()
print(cal.round(3))

## 4. What drives predictions?

In [ ]:
m=models['logistic regression']
coef=pd.Series(m.named_steps['m'].coef_[0],index=m.named_steps['p'].get_feature_names_out())
print(coef.round(3).sort_values())
pi=permutation_importance(models['gradient boosting'],Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
pd.Series(pi.importances_mean,index=X.columns).sort_values().plot.barh(title='Permutation importance (AUC drop), GB, test set'); plt.show()

## 5. Conclusion

**Yes, partly.** Claims are predictable well above chance, but not precisely.

- Held-out test: logistic regression ROC AUC 0.76, PR AUC 0.36 (the base rate is 0.126). Cross-validation on the training set gave AUC 0.72 +/- 0.02, so expect roughly 0.72-0.76 on new data. The test figure is from one split of 1,400 rows and is probably a bit optimistic.
- Gradient boosting did no better than logistic regression (AUC 0.74 test, 0.70 CV), so the signal looks simple and roughly monotone. This is not an exhaustive tuning of either model.
- Practical use is ranking, not classifying. Flagging the riskiest 20% catches about half of the claims (recall 0.49). That group's claim rate is 31%, against 8% for the rest. At that threshold precision is only 0.31, so most flagged warranties still do not claim.
- Probabilities are reasonably calibrated by decile, though the lowest deciles are over-predicted somewhat.
- Drivers (standardized logistic coefficients): older devices, more daily use and higher price all raise claim risk; `partner` channel is higher than `online`/`store`. The data are observational, so these are associations, not causes.

**Caveats**
- 23 rows have `daily_hours` above 24, which is impossible. They were kept and clipped to 24. Worth checking at the source.
- `device_serial` alone gives AUC 0.52. That is about 2 standard errors from 0.5 and the serial is excluded from the models. It is probably noise, but if serials encode batch or production date they could carry real information. Ask the data owner.
- The split is random. There is no date column, so I could not test performance on future sales, which is what deployment would involve.
- Only 4 features are available. More signal (product type, region, claim history) would likely help more than model tuning.
